In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [10]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=tools,
    system_prompt=prompt
)

In [12]:
from langchain.messages import HumanMessage

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]}
)

In [17]:
from pprint import pprint

pprint(response['messages'][-1].content[0]['text'])

('The **`langchain-mcp-adapters`** library is a Python package designed to '
 "bridge Anthropic's **Model Context Protocol (MCP)** with **LangChain** and "
 '**LangGraph**. \n'
 '\n'
 '### Purpose and Features\n'
 '* **Tool Conversion:** It acts as a lightweight adapter layer that converts '
 'MCP tools into standard LangChain-compatible tools (`Tool` objects), '
 'allowing LLMs and AI agents in LangChain/LangGraph to seamlessly invoke '
 'them.\n'
 '* **Client & Server Integration:** It provides client implementations to '
 'connect to single or multiple MCP servers using various communication '
 'transports (such as `stdio`, HTTP, and streamable HTTP).\n'
 '* **Multi-Server Workflows:** It enables agents to aggregate and load tools '
 'across multiple disparate MCP servers simultaneously.\n'
 '\n'
 '---\n'
 '\n'
 '### ⚠️ Important Update: Migration Notice\n'
 'The standalone `langchain-mcp-adapters` library **is no longer actively '
 'maintained** as a separate repository. \n'
 '\n'


## Online MCP

In [26]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [27]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=tools,
)

In [28]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response['messages'][-1].content[0]['text'])

'It is 10:52 PM EST on Wednesday, November 11, 2026.'
